# 35 — AI Governance, Compliance & Risk Frameworks — Lab

## Description

This lab conducts a structured algorithmic audit on a hiring-decision classifier and produces a complete AI Impact Assessment for a high-risk deployment scenario. You will compute accuracy and fairness metrics, review model transparency evidence, fill an audit report template, draft an AIIA, and formulate a deployment recommendation based on audit findings.

## Objectives

- Conduct a structured algorithmic audit of a classifier covering accuracy, fairness, transparency, and accountability dimensions.
- Produce a complete AI Impact Assessment document for a high-risk AI deployment scenario, including system description, risk identification, affected populations, mitigation plan, and residual risk statement.

## Prerequisites

You should have completed the m35 lecture and be familiar with the NIST AI RMF, EU AI Act risk classification, the four audit dimensions (accuracy, fairness, transparency, accountability), and the five AI Impact Assessment components. You should also have basic familiarity with fairness metrics from m32 (Demographic Parity, Equalized Odds) and explainability outputs from m34 (SHAP).

## Required Software/Packages

- Python 3.8+
- pandas
- numpy
- scikit-learn
- aif360
- matplotlib

## Environment/Setup Instructions

All required libraries are pre-installed in the standard environment. No additional setup is required.

## Background/Context

Imagine you are a governance specialist at a mid-sized technology company evaluating a proposed hiring classifier. The model will screen resumes to decide which candidates advance to phone interviews. Your task is to audit the model against four audit dimensions, produce an AI Impact Assessment with five required components, and make a deployment recommendation. The model was trained on 10 years of historical hiring decisions from your company. Approximately 30 percent of the training data comes from applications to engineering roles, 25 percent from sales, 20 percent from product, and 25 percent from other roles.

The model achieves 88 percent overall accuracy on a held-out test set. However, preliminary fairness analysis suggests disparities exist across demographic groups. Your audit will quantify these disparities, assess mitigation strategies, and determine whether the model meets compliance standards before deployment.

## Exercise Instructions

1. Load the hiring classifier and test dataset; compute overall accuracy and accuracy disaggregated by gender and race. (6 min)
2. Compute Demographic Parity difference and Equalized Odds difference using AIF360; load and examine a pre-computed SHAP summary plot. (9 min)
3. Using the audit template provided below, fill in findings for all four dimensions (accuracy, fairness, transparency, accountability); assign a compliance status (pass / flag / fail) per dimension. (9 min)
4. Draft the AI Impact Assessment with all five sections: system description, risk identification, affected populations, mitigation plan, and residual risk statement; conclude with a deployment recommendation (deploy / conditional deploy / do not deploy). (6 min)

In [ ]:
# Starter Code: Import Libraries and Generate Synthetic Hiring Dataset

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from aif360.datasets import StandardDataset
from aif360.metrics import ClassificationMetric
import matplotlib.pyplot as plt
import json

# Generate synthetic hiring dataset
# Attributes: GPA, years_experience, interview_score, gender (0=Female, 1=Male), race (0=Non-White, 1=White)
# Label: hired (0=No, 1=Yes)

np.random.seed(42)
n_samples = 1000

data = {
    'gpa': np.random.uniform(2.5, 4.0, n_samples),
    'years_experience': np.random.randint(0, 20, n_samples),
    'interview_score': np.random.uniform(1, 10, n_samples),
    'gender': np.random.choice([0, 1], n_samples, p=[0.4, 0.6]),  # 40% Female, 60% Male
    'race': np.random.choice([0, 1], n_samples, p=[0.3, 0.7])  # 30% Non-White, 70% White
}

df = pd.DataFrame(data)

# Generate hiring decisions with inherent bias
# Bias: Female candidates and Non-White candidates have lower hire rates for the same qualifications
df['hired'] = (
    (df['gpa'] > 3.2) & 
    (df['interview_score'] > 5.5) &
    (df['years_experience'] > 2) &
    ((df['gender'] == 1) | (df['race'] == 1))  # Bias: Men and White candidates hired more easily
).astype(int)

# Add some stochasticity
noise_idx = np.random.choice(len(df), size=int(0.15 * len(df)), replace=False)
df.loc[noise_idx, 'hired'] = 1 - df.loc[noise_idx, 'hired']

print(f"Dataset shape: {df.shape}")
print(f"Hired rate overall: {df['hired'].mean():.2%}")
print(f"Hired rate by gender (0=Female, 1=Male):")
for g in [0, 1]:
    print(f"  Gender {g}: {df[df['gender'] == g]['hired'].mean():.2%}")
print(f"Hired rate by race (0=Non-White, 1=White):")
for r in [0, 1]:
    print(f"  Race {r}: {df[df['race'] == r]['hired'].mean():.2%}")

# Split into train and test
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)

# Train logistic regression model
X_train = train_df[['gpa', 'years_experience', 'interview_score', 'gender', 'race']]
y_train = train_df['hired']

X_test = test_df[['gpa', 'years_experience', 'interview_score', 'gender', 'race']]
y_test = test_df['hired']

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

# Predictions
y_pred_test = model.predict(X_test_scaled)

print(f"\nModel trained. Test set size: {len(test_df)}")
print(f"Overall test accuracy: {(y_pred_test == y_test).mean():.2%}")

# Store test data and predictions for next steps
test_results = test_df.copy()
test_results['predicted_hired'] = y_pred_test

## Step 1: Compute Accuracy Disaggregated by Protected Group

Now compute accuracy overall and separately for each gender and race group. Report exact percentages.

In [ ]:
# Step 1: Accuracy Analysis (6 min)
# Your task: Compute accuracy overall and disaggregated by gender and race

overall_accuracy = (y_pred_test == y_test).mean()
print(f"Overall accuracy: {overall_accuracy:.4f}")

# Accuracy by gender
print(f"\nAccuracy by gender:")
accuracy_by_gender = {}
for g in [0, 1]:
    mask = test_df['gender'] == g
    acc = (y_pred_test[mask] == y_test[mask]).mean()
    gender_name = "Female" if g == 0 else "Male"
    print(f"  Gender {g} ({gender_name}): {acc:.4f} (n={mask.sum()})")
    accuracy_by_gender[g] = acc

# Accuracy by race
print(f"\nAccuracy by race:")
accuracy_by_race = {}
for r in [0, 1]:
    mask = test_df['race'] == r
    acc = (y_pred_test[mask] == y_test[mask]).mean()
    race_name = "Non-White" if r == 0 else "White"
    print(f"  Race {r} ({race_name}): {acc:.4f} (n={mask.sum()})")
    accuracy_by_race[r] = acc

# Compute disparity
gender_disparity = abs(accuracy_by_gender[0] - accuracy_by_gender[1])
race_disparity = abs(accuracy_by_race[0] - accuracy_by_race[1])

print(f"\nAccuracy disparity by gender: {gender_disparity:.4f}")
print(f"Accuracy disparity by race: {race_disparity:.4f}")

## Step 2: Compute Fairness Metrics and Review SHAP Evidence

Compute Demographic Parity and Equalized Odds differences using AIF360. Then examine a pre-computed SHAP visualization as transparency evidence.

In [ ]:
# Step 2: Fairness Metrics and Transparency Evidence (9 min)
# Your task: Compute Demographic Parity and Equalized Odds differences

# Prepare data for AIF360
test_data_aif = test_results.copy()
test_data_aif['actual'] = y_test.values

# Group 0: Female, Group 1: Male (gender as protected attribute)
print("Demographic Parity Analysis (by gender):")
print("="*50)

# Demographic Parity: proportion of positive predictions should be equal across groups
dem_parity_diff = {}
for g in [0, 1]:
    mask = test_data_aif['gender'] == g
    positive_rate = (test_data_aif.loc[mask, 'predicted_hired'] == 1).mean()
    dem_parity_diff[g] = positive_rate
    gender_name = "Female" if g == 0 else "Male"
    print(f"  Gender {g} ({gender_name}): {positive_rate:.4f} positive predictions")

dem_parity_gap = abs(dem_parity_diff[0] - dem_parity_diff[1])
print(f"  Demographic Parity difference: {dem_parity_gap:.4f}")
print(f"  Threshold: 0.05")
print(f"  Status: {'PASS' if dem_parity_gap <= 0.05 else 'FLAG'}")

# Equalized Odds: TPR and FPR should be equal across groups
print(f"\nEqualized Odds Analysis (by gender):")
print("="*50)

equalized_odds_diffs = {}
for g in [0, 1]:
    mask = test_data_aif['gender'] == g
    actual = test_data_aif.loc[mask, 'actual']
    predicted = test_data_aif.loc[mask, 'predicted_hired']
    
    # True Positive Rate (sensitivity)
    tp = ((predicted == 1) & (actual == 1)).sum()
    fn = ((predicted == 0) & (actual == 1)).sum()
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    # False Positive Rate
    fp = ((predicted == 1) & (actual == 0)).sum()
    tn = ((predicted == 0) & (actual == 0)).sum()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    gender_name = "Female" if g == 0 else "Male"
    print(f"  Gender {g} ({gender_name}): TPR={tpr:.4f}, FPR={fpr:.4f}")
    equalized_odds_diffs[g] = {'tpr': tpr, 'fpr': fpr}

tpr_gap = abs(equalized_odds_diffs[0]['tpr'] - equalized_odds_diffs[1]['tpr'])
fpr_gap = abs(equalized_odds_diffs[0]['fpr'] - equalized_odds_diffs[1]['fpr'])
equalized_odds_gap = max(tpr_gap, fpr_gap)

print(f"\n  TPR difference: {tpr_gap:.4f}")
print(f"  FPR difference: {fpr_gap:.4f}")
print(f"  Equalized Odds difference (max): {equalized_odds_gap:.4f}")
print(f"  Threshold: 0.05")
print(f"  Status: {'PASS' if equalized_odds_gap <= 0.05 else 'FLAG'}")

## Transparency Evidence: SHAP Analysis

Below is a pre-computed summary of SHAP feature importance from the hiring classifier. Feature importance shows which model inputs most influence hiring predictions.

In [ ]:
# Pre-computed SHAP summary (from external analysis)
# This simulates SHAP output from explainability analysis

shap_importance = {
    'interview_score': 0.35,
    'gpa': 0.28,
    'years_experience': 0.18,
    'gender': 0.12,  # Concerning: gender should not be a top feature
    'race': 0.07
}

print("SHAP Feature Importance (Transparency Evidence):")
print("="*50)
for feature, importance in sorted(shap_importance.items(), key=lambda x: x[1], reverse=True):
    print(f"  {feature}: {importance:.2%}")

print(f"\nAnalysis:")
print(f"  Top 3 features account for {sum(list(shap_importance.values())[:3]):.2%} of model variance")
print(f"  Gender feature importance: {shap_importance['gender']:.2%} (concerning)")
print(f"  Race feature importance: {shap_importance['race']:.2%} (acceptable)")
print(f"\nFinding: Gender appears as a top-4 feature, suggesting the model may rely on demographic")
print(f"proxies. This requires investigation of potential indirect discrimination.")

## Step 3: Audit Template — Fill in Your Findings

Use the audit template below to document findings across the four dimensions. Compute exact values and assign compliance status (pass / flag / fail).

In [ ]:
# Step 3: Algorithmic Audit Template (9 min)
# Your task: Fill in the audit report with findings from steps 1-2 above

audit_report = {
    "module_id": "m35",
    "audit_date": "2026-08-20",
    "system_name": "Hiring Decision Classifier",
    "findings": {
        "accuracy_dimension": {
            "findings": f"""Overall accuracy: {overall_accuracy:.2%}
Accuracy by gender (Female): {accuracy_by_gender[0]:.2%}
Accuracy by gender (Male): {accuracy_by_gender[1]:.2%}
Gender accuracy disparity: {gender_disparity:.2%}
Accuracy by race (Non-White): {accuracy_by_race[0]:.2%}
Accuracy by race (White): {accuracy_by_race[1]:.2%}
Race accuracy disparity: {race_disparity:.2%}
Threshold: 5.00%
Assessment: Disparities detected in both gender and race. Female candidates have lower accuracy.
This suggests the model makes different error rates for different demographic groups.""",
            "status": "flag" if max(gender_disparity, race_disparity) > 0.05 else "pass"
        },
        "fairness_dimension": {
            "findings": f"""Demographic Parity difference (gender): {dem_parity_gap:.4f}
Equalized Odds TPR difference: {tpr_gap:.4f}
Equalized Odds FPR difference: {fpr_gap:.4f}
Equalized Odds difference (max): {equalized_odds_gap:.4f}
Threshold: 0.05
Assessment: Both Demographic Parity and Equalized Odds show gaps above acceptable thresholds.
The model systematically produces different outcomes for different demographic groups.""",
            "status": "flag" if (dem_parity_gap > 0.05 or equalized_odds_gap > 0.05) else "pass"
        },
        "transparency_dimension": {
            "findings": f"""SHAP top-3 features: interview_score (35%), gpa (28%), years_experience (18%)
Feature examination: interview_score and gpa are job-related and appropriate.
Years of experience is job-related and appropriate.
Demographic features: gender (12%), race (7%) - both present in top features.
Assessment: Gender ranks as 4th most important feature (12% importance).
This suggests potential use of demographic information (directly or via proxies) in predictions.
Flag for investigation: model may be using demographic proxies against policy.""",
            "status": "flag"
        },
        "accountability_dimension": {
            "findings": f"""Human review process: Weekly manual review of top 100 candidates by senior hiring manager documented.
Appeal procedure: Candidates rejected can submit appeals; reviewed by HR director.
Audit trail: All model decisions logged with timestamp and features used.
Authority: Chief AI Officer approval required before deployment (documented in governance policy).
Escalation: Model performance will be monitored quarterly; triggers retraining if gender disparity exceeds 8%.
Assessment: Organizational safeguards are partially in place but need strengthening.
No independent third-party audit has been conducted yet.""",
            "status": "flag"
        }
    },
    "overall_recommendation": "conditional_deploy",
    "reasoning": """The classifier shows concerning fairness disparities across demographic groups (gender and race).
Demographic Parity and Equalized Odds thresholds are exceeded. Demographic features appear to influence
predictions. However, human review safeguards and audit procedures are documented. Recommend conditional
deployment pending: (1) retrain model with fairness constraints, (2) increase manual review scope to top 500
candidates, (3) conduct third-party algorithmic audit, (4) establish performance monitoring dashboard,
(5) implement bias remediation procedures in HR workflows. Revisit deployment decision in 30 days."""
}

print("ALGORITHMIC AUDIT REPORT")
print("="*70)
print(f"System: {audit_report['system_name']}")
print(f"Audit Date: {audit_report['audit_date']}")
print(f"\n")

for dimension, data in audit_report['findings'].items():
    status_upper = data['status'].upper()
    print(f"\n{dimension.upper().replace('_', ' ')}")
    print(f"Status: [{status_upper}]")
    print(f"{data['findings']}")

print(f"\n\nOVERALL RECOMMENDATION: {audit_report['overall_recommendation'].upper().replace('_', ' ')}")
print(f"\nREASONING:")
print(audit_report['reasoning'])

# Store for next step
audit_findings = audit_report

## Step 4: AI Impact Assessment

Now draft the complete AI Impact Assessment for this hiring system. Include all five required components: system description, risk identification, affected populations, mitigation plan, and residual risk statement. Conclude with a deployment recommendation justified by audit findings.

In [ ]:
# Step 4: AI Impact Assessment (6 min)
# Your task: Draft the complete AIIA with all five components

aiia = {
    "title": "AI Impact Assessment: Hiring Decision Classifier",
    "system_description": """The Hiring Decision Classifier is a logistic regression model trained on 10 years of historical hiring decisions
across 500+ job listings (30% engineering, 25% sales, 20% product, 25% other roles). The model predicts whether a candidate
should advance to phone interview based on resume data (GPA, years of experience, interview score) and automatically categorizes
candidates into interview/no-interview decisions. The system is deployed as part of the resume-screening workflow used by the
recruiting team to process 10,000+ applications per quarter.""",
    
    "risk_identification": [
        {
            "risk": "Demographic bias in predictions",
            "affected_dimension": "fairness",
            "likelihood": "high",
            "severity": "high",
            "evidence": f"Gender accuracy disparity: {gender_disparity:.2%}. Race accuracy disparity: {race_disparity:.2%}. Demographic Parity difference: {dem_parity_gap:.4f}.",
            "residual_risk_after_mitigation": "medium"
        },
        {
            "risk": "Model uses demographic proxies",
            "affected_dimension": "transparency",
            "likelihood": "high",
            "severity": "high",
            "evidence": f"Gender feature ranks 4th in SHAP importance ({shap_importance['gender']:.1%}). Training data reflects historical bias in hiring.",
            "residual_risk_after_mitigation": "medium"
        },
        {
            "risk": "Disparate impact on minority candidates",
            "affected_dimension": "accountability",
            "likelihood": "high",
            "severity": "high",
            "evidence": f"Non-White candidates have {accuracy_by_race[0]:.2%} accuracy vs. {accuracy_by_race[1]:.2%} for White candidates.",
            "residual_risk_after_mitigation": "medium"
        },
        {
            "risk": "Data quality issues",
            "affected_dimension": "accuracy",
            "likelihood": "medium",
            "severity": "medium",
            "evidence": "Training data drawn from historical hiring decisions, which themselves contain bias and may not represent optimal hiring practices.",
            "residual_risk_after_mitigation": "medium"
        }
    ],
    
    "affected_populations": """The model directly affects hiring outcomes for all candidates. Disproportionately affected groups include:
Female candidates: Show 3-5% lower accuracy and lower positive prediction rates than male candidates.
Non-White candidates: Show 4-6% lower accuracy than White candidates.
Candidates from economically disadvantaged backgrounds: May have lower GPA and years of experience due to structural barriers; model amplifies this disadvantage.
These populations face reduced interview opportunities, limiting employment and economic mobility.""",
    
    "mitigation_plan": """Mitigation 1: Model Retraining with Fairness Constraints
Retrain the logistic regression model with demographic parity constraints. Target threshold: gender/race demographic parity difference <= 0.05.
Timeline: 2 weeks. Responsibility: ML Engineering team.

Mitigation 2: Human-in-the-Loop Review
Increase manual human review scope from top 100 to top 500 candidates. Senior hiring managers review flagged predictions.
All interview/no-interview decisions pass through human review before HR sees them.
Timeline: Immediate. Responsibility: Recruiting team (Chief Recruiter).

Mitigation 3: Third-Party Algorithmic Audit
Engage independent external auditor to conduct full algorithmic audit covering accuracy, fairness, transparency, and accountability.
Auditor scope: review model card, conduct statistical testing, verify audit trail completeness.
Timeline: 4 weeks. Responsibility: Chief AI Officer.

Mitigation 4: Monitoring and Escalation
Establish quarterly performance monitoring dashboard tracking gender/race accuracy and fairness metrics across all demographic groups.
Escalation trigger: if demographic parity difference exceeds 8% or accuracy disparity exceeds 7%, halt model deployment and retrain.
Timeline: Ongoing. Responsibility: Data Science team.

Mitigation 5: Bias Remediation Procedures
Establish HR escalation procedure: candidates rejected by model can submit bias appeals. Appeals reviewed within 5 business days.
Candidates may provide additional information (new certifications, projects) for reconsideration.
Timeline: Immediate. Responsibility: Chief HR Officer.""",
    
    "residual_risk_statement": """After implementing mitigations 1-5 above, residual risks remain:

Residual Risk 1: Demographic Parity Gap. After fairness-constrained retraining, target demographic parity difference is <= 0.05.
However, residual gap of 3-5% is expected due to inherent data biases. Risk Level: Medium.

Residual Risk 2: Model still influences recommendation order. Even with human review, the model determines which candidates appear
first in the review list. Hiring managers' unconscious biases may amplify model predictions (anchoring effect).
Risk Level: Medium.

Residual Risk 3: Historical bias embedded in training data. The 10-year training dataset reflects past discriminatory practices.
Even with fairness constraints, the model cannot fully eliminate this historical bias.
Risk Level: Medium.

Residual Risk Mitigation: Quarterly audits and appeals reviews are required to track residual risk over time.
If residual risks exceed stated thresholds after 2 quarters, system will be paused for model redesign or replacement.

EU AI Act Classification: High-Risk (Employment category, Annex III). Mandatory compliance requirements apply.""",
    
    "deployment_recommendation": "CONDITIONAL DEPLOY"
}

print("AI IMPACT ASSESSMENT")
print("="*70)
print(f"Title: {aiia['title']}")
print(f"\n")

print("SYSTEM DESCRIPTION")
print("-"*70)
print(aiia['system_description'])

print(f"\n\nRISK IDENTIFICATION")
print("-"*70)
for i, risk in enumerate(aiia['risk_identification'], 1):
    print(f"\nRisk {i}: {risk['risk']}")
    print(f"  Dimension: {risk['affected_dimension']}")
    print(f"  Likelihood: {risk['likelihood']} | Severity: {risk['severity']}")
    print(f"  Evidence: {risk['evidence']}")
    print(f"  Residual Risk: {risk['residual_risk_after_mitigation']}")

print(f"\n\nAFFECTED POPULATIONS")
print("-"*70)
print(aiia['affected_populations'])

print(f"\n\nMITIGATION PLAN")
print("-"*70)
print(aiia['mitigation_plan'])

print(f"\n\nRESIDUAL RISK STATEMENT")
print("-"*70)
print(aiia['residual_risk_statement'])

print(f"\n\nDEPLOYMENT RECOMMENDATION: {aiia['deployment_recommendation']}")
print("\nJustification: The classifier demonstrates fairness concerns and demographic disparities. However, organizational")
print("safeguards (human review, monitoring, escalation) are documented and feasible to implement within 2-4 weeks.")
print("Conditional deployment with mandatory mitigations enables controlled rollout with oversight. Full deployment is")
print("recommended only after third-party audit confirms fairness constraints are effective and residual disparities")
print("remain within acceptable thresholds.")

## Expected Outputs

- **Step 1 (Accuracy)**: Overall accuracy approximately 88 percent. Accuracy disparity by gender and race should be visible (3-6 percentage points).
- **Step 2 (Fairness & Transparency)**: Demographic Parity difference and Equalized Odds difference both above 0.05 threshold. SHAP shows gender in top-4 features, confirming transparency concern.
- **Step 3 (Audit Report)**: Four dimensions assessed with status for each (pass/flag/fail). Overall recommendation is conditional_deploy.
- **Step 4 (AIIA)**: Five-section AIIA complete with explicit risk statement, affected populations, and mitigations. Deployment recommendation clearly stated.

## Comprehension Questions

1. Based on the audit findings, why does the model's demographic parity difference matter for hiring compliance? What does a 0.08 difference mean in practical terms?
2. In the AIIA, why are "affected populations" documented separately from "risk identification"? What role does naming affected groups play in governance?

In [ ]:
# Verification: Algorithmic Audit Completeness Check
# Verify that audit report contains all four dimensions and at least one status per dimension

required_dimensions = {'accuracy_dimension', 'fairness_dimension', 'transparency_dimension', 'accountability_dimension'}
required_fields = {'findings', 'status'}

print("Verification: Audit Report Completeness")
print("="*50)

audit_complete = True
for dimension in required_dimensions:
    if dimension in audit_findings['findings']:
        data = audit_findings['findings'][dimension]
        has_fields = all(field in data for field in required_fields)
        has_valid_status = data['status'] in ['pass', 'flag', 'fail']
        print(f"  {dimension}: OK (status={data['status']})")
        if not (has_fields and has_valid_status):
            audit_complete = False
    else:
        print(f"  {dimension}: MISSING")
        audit_complete = False

if audit_complete:
    print(f"\nRESULT: PASS — Audit report contains all four dimensions with valid status.")
else:
    print(f"\nRESULT: FAIL — Audit report is incomplete.")

In [ ]:
# Verification: AIIA Completeness Check
# Verify that AIIA contains all five required components

required_components = {
    'system_description': str,
    'risk_identification': list,
    'affected_populations': str,
    'mitigation_plan': str,
    'residual_risk_statement': str
}

print("Verification: AI Impact Assessment Completeness")
print("="*50)

aiia_complete = True
for component, expected_type in required_components.items():
    if component in aiia:
        value = aiia[component]
        is_valid_type = isinstance(value, expected_type)
        is_nonempty = (len(value) > 0) if isinstance(value, (str, list)) else True
        
        if is_valid_type and is_nonempty:
            print(f"  {component}: OK")
        else:
            print(f"  {component}: INCOMPLETE")
            aiia_complete = False
    else:
        print(f"  {component}: MISSING")
        aiia_complete = False

# Check deployment recommendation
if 'deployment_recommendation' in aiia:
    rec = aiia['deployment_recommendation'].upper()
    if rec in ['DEPLOY', 'CONDITIONAL DEPLOY', 'DO NOT DEPLOY']:
        print(f"  deployment_recommendation: OK ({rec})")
    else:
        print(f"  deployment_recommendation: INVALID ({rec})")
        aiia_complete = False
else:
    print(f"  deployment_recommendation: MISSING")
    aiia_complete = False

if aiia_complete:
    print(f"\nRESULT: PASS — AIIA contains all five components with valid deployment recommendation.")
else:
    print(f"\nRESULT: FAIL — AIIA is incomplete.")

## Optional Challenges

**Challenge 1**: Recompute fairness metrics disaggregated by intersecting gender AND race (four groups: female + non-white, female + white, male + non-white, male + white). Do fairness disparities persist or change when you slice the data this way?

**Challenge 2**: Draft a model card for this classifier following the OpenAI model card template. Include sections on model type, training procedure, performance benchmarks disaggregated by demographic group, recommended use cases, and limitations.

**Challenge 3**: The AIIA states that fairness-constrained retraining is mitigation 1. Propose a specific constraint type (Demographic Parity, Equalized Odds, or Calibration) and explain why you chose it over the others for this hiring scenario.

## Completion Criteria

You have completed this lab successfully when:

1. **Step 1**: Accuracy values are computed overall and disaggregated by gender and race, with exact percentages reported.
2. **Step 2**: Demographic Parity and Equalized Odds differences are computed using fairness metrics, and SHAP feature importance is examined for transparency concerns.
3. **Step 3**: Audit report is completed with findings for all four dimensions (accuracy, fairness, transparency, accountability), each with an assigned status (pass/flag/fail). The verification cell prints PASS.
4. **Step 4**: AI Impact Assessment is drafted with all five components (system description, risk identification, affected populations, mitigation plan, residual risk statement) and concludes with a deployment recommendation (deploy / conditional deploy / do not deploy). The verification cell prints PASS.
5. **Comprehension Questions**: You have answered both comprehension questions in writing.
6. **Both verification cells run without error and print PASS** for audit report and AIIA completeness.